# Rolling Market Beta

## Research question

This experiment tests whether trailing market beta contains useful cross-sectional information in the S&P 100 universe.

The primary signal is low beta:

$$
\text{LowBeta}_{i,t}=-\beta_{i,t},
$$

where beta is estimated from trailing stock and SPY returns over 126 trading days.

Intuition: the low-beta anomaly

* Low-beta stocks earn lower raw returns, but more return than CAPM predicts.
* High-beta stocks earn higher raw returns, but less return than CAPM predicts.
* Consequently, low-beta stocks may offer higher alpha or Sharpe ratios despite lower absolute returns.

## Signal construction

In [1]:
import numpy as np
import pandas as pd

from alpha_research.config.paths import PROCESSED_DATA_DIR
from alpha_research.signal_processing import (
    add_sector_neutral_factor,
    process_factor_columns,
)
from alpha_research.validation import calculate_ic_by_horizon

factor_panel = pd.read_parquet(
    PROCESSED_DATA_DIR / "factor_panel.parquet"
)

required_columns = [
    "date",
    "ticker",
    "sector",
    "beta_126",
    "realised_vol_63_raw",
    "idio_vol_63_raw",
    "mom_12_1m_z",
    "forward_ret_1d",
    "forward_ret_5d",
]

missing_columns = [
    column
    for column in required_columns
    if column not in factor_panel.columns
]

assert not missing_columns, f"Missing columns: {missing_columns}"

beta_panel = (
    factor_panel
    .sort_values(["ticker", "date"])
    .reset_index(drop=True)
    .copy()
)

beta_panel[required_columns].info()

<class 'pandas.DataFrame'>
RangeIndex: 284249 entries, 0 to 284248
Data columns (total 9 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   date                 284249 non-null  datetime64[ms]
 1   ticker               284249 non-null  str           
 2   sector               284249 non-null  str           
 3   beta_126             277936 non-null  float64       
 4   realised_vol_63_raw  277936 non-null  float64       
 5   idio_vol_63_raw      277936 non-null  float64       
 6   mom_12_1m_z          259036 non-null  float64       
 7   forward_ret_1d       284148 non-null  float64       
 8   forward_ret_5d       283744 non-null  float64       
dtypes: datetime64[ms](1), float64(6), str(2)
memory usage: 24.5 MB


In [2]:
beta_panel["low_beta_raw"] = (
    -beta_panel["beta_126"]
    .replace([np.inf, -np.inf], np.nan)
)

beta_distribution_summary = (
    beta_panel[
        [
            "beta_126",
            "low_beta_raw",
        ]
    ]
    .agg(
        [
            "count",
            "mean",
            "std",
            "min",
            "median",
            "max",
            "skew",
        ]
    )
    .T
)

beta_distribution_summary["coverage"] = (
    beta_panel[
        [
            "beta_126",
            "low_beta_raw",
        ]
    ]
    .notna()
    .mean()
)

beta_distribution_summary

,count,mean,std,min,median,max,skew,coverage
beta_126,277936.0,0.967115,0.487243,-0.817177,0.964061,3.794389,0.477018,0.977791
low_beta_raw,277936.0,-0.967115,0.487243,-3.794389,-0.964061,0.817177,-0.477018,0.977791


In [3]:
beta_panel["beta_126"].quantile(
    [0.001, 0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99, 0.999]
)

0.001   -0.438838
0.010   -0.107661
0.050    0.197103
0.250    0.652649
0.500    0.964061
0.750    1.239566
0.950    1.793544
0.990    2.416792
0.999    3.018982
Name: beta_126, dtype: float64

### Process and sector-neutralise

In [4]:
beta_panel = process_factor_columns(
    beta_panel,
    factor_map={
        "low_beta_raw": "low_beta",
    },
    lower_quantile=0.01,
    upper_quantile=0.99,
)

beta_panel = add_sector_neutral_factor(
    beta_panel,
    factor_column="low_beta_winsorised",
    output_column="low_beta_sector_neutral_z",
    sector_column="sector",
    min_sector_observations=3,
)

### Initial IC screen

In [5]:
beta_signals = {
    "Low beta — raw": "low_beta_z",
    "Low beta — sector neutral": (
        "low_beta_sector_neutral_z"
    ),
}

beta_ic_results = []

for signal_name, factor_column in beta_signals.items():
    result = calculate_ic_by_horizon(
        panel=beta_panel,
        factor_column=factor_column,
        forward_return_columns=[
            "forward_ret_1d",
            "forward_ret_5d",
        ],
        method="spearman",
        min_observations=30,
    )

    result["signal"] = signal_name
    beta_ic_results.append(result)

beta_ic_summary = pd.concat(
    beta_ic_results,
    ignore_index=True,
)

beta_ic_summary[
    [
        "signal",
        "forward_return_column",
        "count",
        "mean_ic",
        "std_ic",
        "ic_ir",
        "t_stat",
        "positive_fraction",
    ]
]

,signal,forward_return_column,count,mean_ic,std_ic,ic_ir,t_stat,positive_fraction
0,Low beta — raw,forward_ret_1d,2827.0,-0.011704,0.351269,-0.033318,-1.771512,0.477184
1,Low beta — raw,forward_ret_5d,2823.0,-0.029633,0.344032,-0.086135,-4.576543,0.461211
2,Low beta — sector neutral,forward_ret_1d,2827.0,-0.007920,0.221011,-0.035833,-1.905233,0.476123
3,Low beta — sector neutral,forward_ret_5d,2823.0,-0.021906,0.216274,-0.101290,-5.381726,0.439249


### Redundancy check

In [6]:
beta_correlation_pairs = {
    "Low beta vs realised volatility": (
        "low_beta_z",
        "realised_vol_63_raw",
    ),
    "Low beta vs idiosyncratic volatility": (
        "low_beta_z",
        "idio_vol_63_raw",
    ),
    "Low beta vs momentum": (
        "low_beta_z",
        "mom_12_1m_z",
    ),
}

beta_correlation_rows = []

for date, group in beta_panel.groupby("date"):
    for pair_name, (left_column, right_column) in (
        beta_correlation_pairs.items()
    ):
        valid = group[
            [left_column, right_column]
        ].dropna()

        if len(valid) < 30:
            continue

        beta_correlation_rows.append(
            {
                "date": date,
                "pair": pair_name,
                "correlation": valid[left_column].corr(
                    valid[right_column],
                    method="spearman",
                ),
            }
        )

daily_beta_correlations = pd.DataFrame(
    beta_correlation_rows
)

beta_correlation_summary = (
    daily_beta_correlations
    .groupby("pair")["correlation"]
    .agg(
        count="count",
        mean="mean",
        median="median",
        std="std",
        min="min",
        max="max",
    )
)

beta_correlation_summary

,count,mean,median,std,min,max
pair,,,,,,
Low beta vs idiosyncratic volatility,2828,-0.407714,-0.417907,0.127536,-0.678009,-0.031441
Low beta vs momentum,2639,-0.147247,-0.272492,0.415025,-0.724003,0.842486
Low beta vs realised volatility,2828,-0.697259,-0.696908,0.131282,-0.963634,-0.322152


### Residualise against realised volatility

In [7]:
required_incremental_columns = [
    "realised_vol_63_z",
    "realised_vol_63_sector_neutral_z",
]

missing_columns = [
    column
    for column in required_incremental_columns
    if column not in beta_panel.columns
]

assert not missing_columns, f"Missing columns: {missing_columns}"

In [8]:
def cross_sectional_rank_residual(
    group,
    target_column,
    explanatory_column,
):
    result = pd.Series(
        np.nan,
        index=group.index,
        dtype=float,
    )

    valid = group[
        [target_column, explanatory_column]
    ].dropna()

    if len(valid) < 30:
        return result

    target_rank = valid[target_column].rank(
        method="average",
        pct=True,
    )
    explanatory_rank = valid[explanatory_column].rank(
        method="average",
        pct=True,
    )

    target_standardised = (
        target_rank - target_rank.mean()
    ) / target_rank.std(ddof=0)

    explanatory_standardised = (
        explanatory_rank - explanatory_rank.mean()
    ) / explanatory_rank.std(ddof=0)

    design_matrix = np.column_stack(
        [
            np.ones(len(valid)),
            explanatory_standardised.to_numpy(),
        ]
    )

    coefficients = np.linalg.lstsq(
        design_matrix,
        target_standardised.to_numpy(),
        rcond=None,
    )[0]

    result.loc[valid.index] = (
        target_standardised.to_numpy()
        - design_matrix @ coefficients
    )

    return result

In [9]:
beta_panel["low_beta_incremental_to_realised_vol"] = (
    beta_panel
    .groupby("date", group_keys=False)
    .apply(
        lambda group: cross_sectional_rank_residual(
            group=group,
            target_column="low_beta_z",
            explanatory_column="realised_vol_63_z",
        ),
    )
)

beta_panel[
    "low_beta_sector_incremental_to_realised_vol"
] = (
    beta_panel
    .groupby("date", group_keys=False)
    .apply(
        lambda group: cross_sectional_rank_residual(
            group=group,
            target_column="low_beta_sector_neutral_z",
            explanatory_column=(
                "realised_vol_63_sector_neutral_z"
            ),
        ),
    )
)

In [10]:
incremental_beta_signals = {
    "Low beta incremental to realised vol": (
        "low_beta_incremental_to_realised_vol"
    ),
    "Low beta sector-neutral incremental to realised vol": (
        "low_beta_sector_incremental_to_realised_vol"
    ),
}

incremental_beta_results = []

for signal_name, factor_column in (
    incremental_beta_signals.items()
):
    result = calculate_ic_by_horizon(
        panel=beta_panel,
        factor_column=factor_column,
        forward_return_columns=[
            "forward_ret_1d",
            "forward_ret_5d",
        ],
        method="spearman",
        min_observations=30,
    )

    result["signal"] = signal_name
    incremental_beta_results.append(result)

incremental_beta_ic_summary = pd.concat(
    incremental_beta_results,
    ignore_index=True,
)

incremental_beta_ic_summary[
    [
        "signal",
        "forward_return_column",
        "count",
        "mean_ic",
        "std_ic",
        "ic_ir",
        "t_stat",
        "positive_fraction",
    ]
]

,signal,forward_return_column,count,mean_ic,std_ic,ic_ir,t_stat,positive_fraction
0,Low beta incremental to realised vol,forward_ret_1d,2827.0,-0.008864,0.248631,-0.035651,-1.895521,0.474354
1,Low beta incremental to realised vol,forward_ret_5d,2823.0,-0.013988,0.242966,-0.057573,-3.058987,0.476443
2,Low beta sector-neutral incremental to realise...,forward_ret_1d,2827.0,-0.006454,0.168723,-0.038252,-2.033863,0.481783
3,Low beta sector-neutral incremental to realise...,forward_ret_5d,2823.0,-0.012270,0.165671,-0.074060,-3.934967,0.460503


### Quantile shape

In [11]:
def assign_daily_quintiles(series):
    result = pd.Series(
        pd.NA,
        index=series.index,
        dtype="Int64",
    )

    valid = series.dropna()

    if len(valid) < 30:
        return result

    percentile_rank = valid.rank(
        method="first",
        pct=True,
    )

    result.loc[valid.index] = (
        np.ceil(percentile_rank * 5)
        .clip(1, 5)
        .astype("Int64")
    )

    return result


beta_panel["beta_quintile"] = (
    beta_panel
    .groupby("date")["beta_126"]
    .transform(assign_daily_quintiles)
)

In [12]:
beta_quintile_results = []

for return_column in [
    "forward_ret_1d",
    "forward_ret_5d",
]:
    daily_quintile_returns = (
        beta_panel
        .dropna(
            subset=[
                "beta_quintile",
                return_column,
            ]
        )
        .groupby(
            ["date", "beta_quintile"],
            observed=True,
        )[return_column]
        .mean()
        .rename("quintile_return")
        .reset_index()
    )

    quintile_summary = (
        daily_quintile_returns
        .groupby(
            "beta_quintile",
            observed=True,
        )["quintile_return"]
        .agg(
            count="count",
            mean_return="mean",
            std_return="std",
            positive_fraction=lambda x: (x > 0).mean(),
        )
        .reset_index()
    )

    quintile_summary["forward_return_column"] = (
        return_column
    )

    beta_quintile_results.append(quintile_summary)

beta_quintile_summary = pd.concat(
    beta_quintile_results,
    ignore_index=True,
)

beta_quintile_summary[
    [
        "forward_return_column",
        "beta_quintile",
        "count",
        "mean_return",
        "std_return",
        "positive_fraction",
    ]
]

,forward_return_column,beta_quintile,count,mean_return,std_return,positive_fraction
0,forward_ret_1d,1,2827,0.000467,0.008583,0.548992
1,forward_ret_1d,2,2827,0.000506,0.010080,0.546869
2,forward_ret_1d,3,2827,0.000595,0.011722,0.545455
3,forward_ret_1d,4,2827,0.000909,0.013673,0.562080
4,forward_ret_1d,5,2827,0.001173,0.018729,0.556774
5,forward_ret_5d,1,2823,0.002271,0.017522,0.589444
6,forward_ret_5d,2,2823,0.002548,0.021004,0.604676
7,forward_ret_5d,3,2823,0.002788,0.025096,0.589090
8,forward_ret_5d,4,2823,0.004342,0.028648,0.619554
9,forward_ret_5d,5,2823,0.006009,0.039572,0.599362


In [13]:
beta_spread_results = []

for return_column in [
    "forward_ret_1d",
    "forward_ret_5d",
]:
    daily_quintile_returns = (
        beta_panel
        .dropna(
            subset=[
                "beta_quintile",
                return_column,
            ]
        )
        .groupby(
            ["date", "beta_quintile"],
            observed=True,
        )[return_column]
        .mean()
        .unstack()
    )

    high_minus_low = (
        daily_quintile_returns[5]
        - daily_quintile_returns[1]
    ).dropna()

    beta_spread_results.append(
        {
            "forward_return_column": return_column,
            "count": high_minus_low.count(),
            "mean_high_minus_low": high_minus_low.mean(),
            "std": high_minus_low.std(),
            "positive_fraction": (
                high_minus_low > 0
            ).mean(),
        }
    )

beta_quintile_spread_summary = pd.DataFrame(
    beta_spread_results
)

beta_quintile_spread_summary

,forward_return_column,count,mean_high_minus_low,std,positive_fraction
0,forward_ret_1d,2827,0.000706,0.017450,0.533428
1,forward_ret_5d,2823,0.003739,0.038234,0.542685


## Portfolio-level test

Construct four portfolios:

* Q1: equal-weight lowest-beta quintile
* Q5: equal-weight highest-beta quintile
* Low-minus-high: $R_{Q1}-R_{Q5}$
* Beta-matched BAB portfolio:

$$
R_{\text{BAB},t}
=
\frac{R_{Q1,t}-R_{f,t}}{\beta_{Q1,t-1}}
-
\frac{R_{Q5,t}-R_{f,t}}{\beta_{Q5,t-1}}.
$$


### Construct daily Q1 and Q5 portfolios

In [14]:
required_bab_columns = [
    "date",
    "ticker",
    "beta_126",
    "beta_quintile",
    "forward_ret_1d",
]

missing_columns = [
    column
    for column in required_bab_columns
    if column not in beta_panel.columns
]

assert not missing_columns, f"Missing columns: {missing_columns}"

assert not beta_panel.duplicated(
    ["date", "ticker"]
).any(), "Duplicate date–ticker observations found"

In [15]:
extreme_quintile_daily = (
    beta_panel
    .loc[
        beta_panel["beta_quintile"].isin([1, 5]),
        required_bab_columns,
    ]
    .dropna(
        subset=[
            "beta_126",
            "beta_quintile",
            "forward_ret_1d",
        ]
    )
    .groupby(
        ["date", "beta_quintile"],
        observed=True,
    )
    .agg(
        portfolio_return=("forward_ret_1d", "mean"),
        ex_ante_beta=("beta_126", "mean"),
        stock_count=("ticker", "nunique"),
    )
    .reset_index()
)

bab_construction_panel = (
    extreme_quintile_daily
    .pivot(
        index="date",
        columns="beta_quintile",
        values=[
            "portfolio_return",
            "ex_ante_beta",
            "stock_count",
        ],
    )
)

bab_construction_panel.columns = [
    f"{metric}_q{int(quintile)}"
    for metric, quintile in bab_construction_panel.columns
]

bab_construction_panel = (
    bab_construction_panel
    .dropna()
    .sort_index()
)

bab_construction_panel[
    "low_minus_high_return"
] = (
    bab_construction_panel["portfolio_return_q1"]
    - bab_construction_panel["portfolio_return_q5"]
)

bab_construction_panel.head()

,portfolio_return_q1,portfolio_return_q5,ex_ante_beta_q1,ex_ante_beta_q5,stock_count_q1,stock_count_q5,low_minus_high_return
date,,,,,,,
2015-04-06,-0.004396,-0.000643,0.715915,1.397268,19.0,20.0,-0.003753
2015-04-07,0.001513,0.007832,0.717615,1.395180,19.0,20.0,-0.006319
2015-04-08,-0.000633,0.008291,0.716589,1.397408,19.0,20.0,-0.008924
2015-04-09,0.003314,0.004914,0.713247,1.399426,19.0,20.0,-0.001599
2015-04-10,-0.006183,-0.001221,0.712089,1.396900,19.0,20.0,-0.004962


### Audit portfolio sizes and beta estimates

In [16]:
construction_summary = (
    bab_construction_panel[
        [
            "stock_count_q1",
            "stock_count_q5",
            "ex_ante_beta_q1",
            "ex_ante_beta_q5",
            "low_minus_high_return",
        ]
    ]
    .agg(
        [
            "count",
            "mean",
            "std",
            "min",
            "median",
            "max",
        ]
    )
    .T
)

construction_summary

,count,mean,std,min,median,max
stock_count_q1,2827.0,19.178281,0.382817,19.000000,19.000000,20.000000
stock_count_q5,2827.0,20.000000,0.000000,20.000000,20.000000,20.000000
ex_ante_beta_q1,2827.0,0.379681,0.228257,-0.281036,0.409562,0.774949
ex_ante_beta_q5,2827.0,1.638104,0.183760,1.250187,1.633828,2.115404
low_minus_high_return,2827.0,-0.000706,0.017450,-0.129207,-0.000938,0.092264


In [17]:
bab_construction_panel["low_leg_multiplier"] = (
    1.0 / bab_construction_panel["ex_ante_beta_q1"]
)

bab_construction_panel["high_leg_multiplier"] = (
    1.0 / bab_construction_panel["ex_ante_beta_q5"]
)

bab_construction_panel["gross_exposure"] = (
    bab_construction_panel["low_leg_multiplier"].abs()
    + bab_construction_panel["high_leg_multiplier"].abs()
)

leverage_summary = (
    bab_construction_panel[
        [
            "low_leg_multiplier",
            "high_leg_multiplier",
            "gross_exposure",
        ]
    ]
    .agg(
        [
            "count",
            "mean",
            "std",
            "min",
            "median",
            "max",
        ]
    )
    .T
)

leverage_summary

,count,mean,std,min,median,max
low_leg_multiplier,2827.0,0.517738,35.967543,-1688.408567,2.181143,154.149364
high_leg_multiplier,2827.0,0.618285,0.070465,0.472723,0.612060,0.799880
gross_exposure,2827.0,7.056792,35.379755,2.037351,3.035823,1688.978819


In [18]:
unstable_beta_dates = (
    bab_construction_panel
    .loc[
        (bab_construction_panel["ex_ante_beta_q1"] <= 0.20)
        | (bab_construction_panel["ex_ante_beta_q5"] <= 0.20)
        | (bab_construction_panel["low_leg_multiplier"].abs() > 3.0)
        | (bab_construction_panel["high_leg_multiplier"].abs() > 3.0)
    ]
    [
        [
            "ex_ante_beta_q1",
            "ex_ante_beta_q5",
            "low_leg_multiplier",
            "high_leg_multiplier",
            "gross_exposure",
        ]
    ]
)

print("Number of unstable dates:", len(unstable_beta_dates))
unstable_beta_dates.head(10)

Number of unstable dates: 911


,ex_ante_beta_q1,ex_ante_beta_q5,low_leg_multiplier,high_leg_multiplier,gross_exposure
date,,,,,
2017-06-09,0.331047,1.890957,3.020715,0.528833,3.549548
2017-06-13,0.311976,1.894335,3.205377,0.527890,3.733267
2017-06-14,0.312886,1.891341,3.196050,0.528725,3.724776
2017-06-15,0.299506,1.908544,3.338831,0.523960,3.862790
2017-06-16,0.272034,1.954373,3.676009,0.511673,4.187682
2017-06-19,0.264131,1.952607,3.785996,0.512136,4.298131
2017-06-20,0.270706,1.920112,3.694044,0.520803,4.214848
2017-06-21,0.270719,1.918271,3.693863,0.521303,4.215166
2017-06-22,0.271944,1.908171,3.677227,0.524062,4.201289


### Cap and regularise unstable beta dates

In [ ]:
BETA_WEIGHT = 0.60
MAX_GROSS_EXPOSURE = 3.0

for quintile in [1, 5]:
    raw_beta_column = f"ex_ante_beta_q{quintile}"
    regularised_beta_column = f"regularised_beta_q{quintile}"

    bab_construction_panel[regularised_beta_column] = (
        BETA_WEIGHT * bab_construction_panel[raw_beta_column]
        + (1.0 - BETA_WEIGHT) * 1.0
    )

assert (
    (bab_construction_panel[["regularised_beta_q1", "regularised_beta_q5"]] > 0)
    .all()
    .all()
)


In [ ]:
bab_construction_panel["uncapped_low_multiplier"] = (
    1.0 / bab_construction_panel["regularised_beta_q1"]
)

bab_construction_panel["uncapped_high_multiplier"] = (
    1.0 / bab_construction_panel["regularised_beta_q5"]
)

bab_construction_panel["uncapped_gross_exposure"] = (
    bab_construction_panel["uncapped_low_multiplier"]
    + bab_construction_panel["uncapped_high_multiplier"]
)


In [ ]:
bab_construction_panel["gross_scaling_factor"] = (
    MAX_GROSS_EXPOSURE / bab_construction_panel["uncapped_gross_exposure"]
).clip(upper=1.0)

bab_construction_panel["low_leg_multiplier_regularised"] = (
    bab_construction_panel["uncapped_low_multiplier"]
    * bab_construction_panel["gross_scaling_factor"]
)

bab_construction_panel["high_leg_multiplier_regularised"] = (
    bab_construction_panel["uncapped_high_multiplier"]
    * bab_construction_panel["gross_scaling_factor"]
)

bab_construction_panel["regularised_gross_exposure"] = (
    bab_construction_panel["low_leg_multiplier_regularised"]
    + bab_construction_panel["high_leg_multiplier_regularised"]
)


In [ ]:
bab_construction_panel["matched_low_beta"] = (
    bab_construction_panel["low_leg_multiplier_regularised"]
    * bab_construction_panel["regularised_beta_q1"]
)

bab_construction_panel["matched_high_beta"] = (
    bab_construction_panel["high_leg_multiplier_regularised"]
    * bab_construction_panel["regularised_beta_q5"]
)

bab_construction_panel["regularised_net_beta"] = (
    bab_construction_panel["matched_low_beta"]
    - bab_construction_panel["matched_high_beta"]
)


In [ ]:
bab_construction_panel["raw_estimate_net_beta"] = (
    bab_construction_panel["low_leg_multiplier_regularised"]
    * bab_construction_panel["ex_ante_beta_q1"]
    - bab_construction_panel["high_leg_multiplier_regularised"]
    * bab_construction_panel["ex_ante_beta_q5"]
)


In [24]:
regularised_leverage_summary = (
    bab_construction_panel[
        [
            "regularised_beta_q1",
            "regularised_beta_q5",
            "uncapped_gross_exposure",
            "gross_scaling_factor",
            "low_leg_multiplier_regularised",
            "high_leg_multiplier_regularised",
            "regularised_gross_exposure",
            "regularised_net_beta",
            "raw_estimate_net_beta",
        ]
    ]
    .describe(
        percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
    )
    .T
)

regularised_leverage_summary

,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max
regularised_beta_q1,2827.0,6.278083e-01,1.369543e-01,2.313784e-01,2.804526e-01,3.393069e-01,0.575196,0.645737,0.707672,8.312040e-01,8.448994e-01,8.649692e-01
regularised_beta_q5,2827.0,1.382863e+00,1.102562e-01,1.150112e+00,1.162050e+00,1.184157e+00,1.300908,1.380297,1.452188,1.582765e+00,1.613689e+00,1.669242e+00
uncapped_gross_exposure,2827.0,2.426934e+00,4.743135e-01,1.976930e+00,2.000111e+00,2.016455e+00,2.165420,2.262127,2.436349,3.583529e+00,4.189157e+00,4.939436e+00
gross_scaling_factor,2827.0,9.828828e-01,5.636766e-02,6.073568e-01,7.161365e-01,8.371641e-01,1.000000,1.000000,1.000000,1.000000e+00,1.000000e+00,1.000000e+00
low_leg_multiplier_regularised,2827.0,1.644817e+00,3.681260e-01,1.156111e+00,1.183573e+00,1.203074e+00,1.413083,1.548618,1.738537,2.466997e+00,2.553995e+00,2.624950e+00
high_leg_multiplier_regularised,2827.0,7.168814e-01,8.154144e-02,3.750500e-01,4.460054e-01,5.330025e-01,0.687146,0.724209,0.768694,8.444826e-01,8.605483e-01,8.694802e-01
regularised_gross_exposure,2827.0,2.361698e+00,2.994214e-01,1.976930e+00,2.000111e+00,2.016455e+00,2.165420,2.262127,2.436349,3.000000e+00,3.000000e+00,3.000000e+00
regularised_net_beta,2827.0,-2.749049e-19,6.027568e-17,-2.220446e-16,-1.110223e-16,-1.110223e-16,0.000000,0.000000,0.000000,1.110223e-16,1.110223e-16,2.220446e-16
raw_estimate_net_beta,2827.0,-6.186237e-01,2.941491e-01,-1.499933e+00,-1.405326e+00,-1.289330e+00,-0.697149,-0.557462,-0.425231,-2.380738e-01,-2.173509e-01,-1.972750e-01


In [ ]:
cap_diagnostics = pd.Series(
    {
        "total_dates": len(bab_construction_panel),
        "capped_dates": (bab_construction_panel["gross_scaling_factor"] < 1.0).sum(),
        "capped_fraction": (
            bab_construction_panel["gross_scaling_factor"] < 1.0
        ).mean(),
        "negative_regularised_beta_dates": (
            (bab_construction_panel["regularised_beta_q1"] <= 0)
            | (bab_construction_panel["regularised_beta_q5"] <= 0)
        ).sum(),
        "maximum_regularised_gross": (
            bab_construction_panel["regularised_gross_exposure"].max()
        ),
        "maximum_abs_regularised_net_beta": (
            bab_construction_panel["regularised_net_beta"].abs().max()
        ),
    },
    name="value",
)

cap_diagnostics


total_dates                         2.827000e+03
capped_dates                        3.380000e+02
capped_fraction                     1.195614e-01
negative_regularised_beta_dates     0.000000e+00
maximum_regularised_gross           3.000000e+00
maximum_abs_regularised_net_beta    2.220446e-16
Name: value, dtype: float64

### Construct gross BAB returns

#### Download and align the risk-free rate

In [26]:
import yfinance as yf
from pandas.tseries.offsets import BDay

start_date = (
    bab_construction_panel.index.min() - BDay(10)
).strftime("%Y-%m-%d")

end_date = (
    bab_construction_panel.index.max() + BDay(10)
).strftime("%Y-%m-%d")

# The 13-week Treasury bill yield (^IRX)
irx_data = yf.download(
    "^IRX",
    start=start_date,
    end=end_date,
    auto_adjust=False,
    progress=False,
)

if isinstance(irx_data.columns, pd.MultiIndex):
    irx_close = irx_data["Close"].squeeze()
else:
    irx_close = irx_data["Close"]

irx_close.index = pd.to_datetime(irx_close.index).tz_localize(None)
irx_close = irx_close.sort_index()

In [27]:
risk_free_daily = (
    (1.0 + irx_close / 100.0) ** (1.0 / 252.0)
    - 1.0
)

bab_construction_panel["risk_free_rate"] = (
    risk_free_daily
    .reindex(bab_construction_panel.index)
    .ffill()
)

assert (
    bab_construction_panel["risk_free_rate"]
    .notna()
    .all()
), "Missing risk-free rates"

In [28]:
risk_free_summary = (
    bab_construction_panel["risk_free_rate"]
    .describe()
)

risk_free_summary

count    2827.000000
mean        0.000081
std         0.000073
min        -0.000004
25%         0.000009
50%         0.000064
75%         0.000157
max         0.000207
Name: risk_free_rate, dtype: float64

#### Calculate the leg excess returns

In [29]:
bab_construction_panel["q1_excess_return"] = (
    bab_construction_panel["portfolio_return_q1"]
    - bab_construction_panel["risk_free_rate"]
)

bab_construction_panel["q5_excess_return"] = (
    bab_construction_panel["portfolio_return_q5"]
    - bab_construction_panel["risk_free_rate"]
)

In [ ]:
bab_construction_panel["bab_gross_return"] = (
    bab_construction_panel["low_leg_multiplier_regularised"]
    * bab_construction_panel["q1_excess_return"]
    - bab_construction_panel["high_leg_multiplier_regularised"]
    * bab_construction_panel["q5_excess_return"]
)


In [ ]:
bab_construction_panel["bab_uncapped_gross_return"] = (
    bab_construction_panel["uncapped_low_multiplier"]
    * bab_construction_panel["q1_excess_return"]
    - bab_construction_panel["uncapped_high_multiplier"]
    * bab_construction_panel["q5_excess_return"]
)


#### Sanity checks

In [32]:
assert np.isfinite(
    bab_construction_panel[
        [
            "bab_gross_return",
            "bab_uncapped_gross_return",
        ]
    ].to_numpy()
).all()

bab_return_diagnostics = (
    bab_construction_panel[
        [
            "portfolio_return_q1",
            "portfolio_return_q5",
            "low_minus_high_return",
            "bab_gross_return",
            "bab_uncapped_gross_return",
        ]
    ]
    .describe(
        percentiles=[
            0.01,
            0.05,
            0.50,
            0.95,
            0.99,
        ]
    )
    .T
)

bab_return_diagnostics

,count,mean,std,min,1%,5%,50%,95%,99%,max
portfolio_return_q1,2827.0,0.000467,0.008583,-0.085770,-0.022930,-0.012028,0.000632,0.012212,0.020147,0.086818
portfolio_return_q5,2827.0,0.001173,0.018729,-0.158825,-0.049157,-0.029867,0.001733,0.028481,0.049626,0.153146
low_minus_high_return,2827.0,-0.000706,0.017450,-0.129207,-0.047423,-0.028288,-0.000938,0.027890,0.046899,0.092264
bab_gross_return,2827.0,-0.000238,0.015845,-0.079919,-0.043185,-0.025713,-0.000209,0.024769,0.044723,0.101272
bab_uncapped_gross_return,2827.0,-0.000269,0.016957,-0.093340,-0.047885,-0.026557,-0.000219,0.025507,0.049707,0.101272


In [ ]:
cap_effect_summary = pd.Series(
    {
        "return_correlation": (
            bab_construction_panel[
                [
                    "bab_gross_return",
                    "bab_uncapped_gross_return",
                ]
            ]
            .corr()
            .iloc[0, 1]
        ),
        "mean_daily_capped": (bab_construction_panel["bab_gross_return"].mean()),
        "mean_daily_uncapped": (
            bab_construction_panel["bab_uncapped_gross_return"].mean()
        ),
        "mean_absolute_cap_effect": (
            (
                bab_construction_panel["bab_gross_return"]
                - bab_construction_panel["bab_uncapped_gross_return"]
            )
            .abs()
            .mean()
        ),
    },
    name="value",
)

cap_effect_summary


return_correlation          0.992530
mean_daily_capped          -0.000238
mean_daily_uncapped        -0.000269
mean_absolute_cap_effect    0.000453
Name: value, dtype: float64

### CAPM regression

#### Align SPY returns

In [ ]:
import statsmodels.api as sm
import yfinance as yf

spy_start_date = (bab_construction_panel.index.min() - BDay(10)).strftime("%Y-%m-%d")

spy_end_date = (bab_construction_panel.index.max() + BDay(10)).strftime("%Y-%m-%d")

spy_data = yf.download(
    "SPY",
    start=spy_start_date,
    end=spy_end_date,
    auto_adjust=True,
    progress=False,
)

if isinstance(spy_data.columns, pd.MultiIndex):
    spy_close = spy_data["Close"].squeeze()
else:
    spy_close = spy_data["Close"]

spy_close.index = pd.to_datetime(spy_close.index).tz_localize(None)

spy_close = spy_close.sort_index()

spy_forward_return = spy_close.shift(-1) / spy_close - 1.0

bab_construction_panel["spy_forward_return"] = spy_forward_return.reindex(
    bab_construction_panel.index
)

assert (
    bab_construction_panel["spy_forward_return"].notna().all()
), "Missing aligned SPY returns"

bab_construction_panel["spy_excess_return"] = (
    bab_construction_panel["spy_forward_return"]
    - bab_construction_panel["risk_free_rate"]
)


In [35]:
market_return_summary = (
    bab_construction_panel[
        [
            "spy_forward_return",
            "spy_excess_return",
            "risk_free_rate",
        ]
    ]
    .describe()
    .T
)

market_return_summary

,count,mean,std,min,25%,50%,75%,max
spy_forward_return,2827.0,0.000580,0.011165,-0.109424,-0.003659,0.000677,0.005921,0.105019
spy_excess_return,2827.0,0.000499,0.011165,-0.109433,-0.003728,0.000585,0.005799,0.104857
risk_free_rate,2827.0,0.000081,0.000073,-0.000004,0.000009,0.000064,0.000157,0.000207


#### Preliminary gross performance

In [ ]:
gross_return_series = {
    "Q1 low beta": bab_construction_panel["portfolio_return_q1"],
    "Q5 high beta": bab_construction_panel["portfolio_return_q5"],
    "Low minus high": bab_construction_panel["low_minus_high_return"],
    "BAB capped": bab_construction_panel["bab_gross_return"],
    "BAB uncapped": bab_construction_panel["bab_uncapped_gross_return"],
}

sharpe_return_series = {
    "Q1 low beta": bab_construction_panel["q1_excess_return"],
    "Q5 high beta": bab_construction_panel["q5_excess_return"],
    "Low minus high": bab_construction_panel["low_minus_high_return"],
    "BAB capped": bab_construction_panel["bab_gross_return"],
    "BAB uncapped": bab_construction_panel["bab_uncapped_gross_return"],
}


In [37]:
gross_performance_rows = []

for portfolio_name, returns in gross_return_series.items():
    returns = returns.dropna()
    sharpe_returns = (
        sharpe_return_series[portfolio_name]
        .reindex(returns.index)
        .dropna()
    )

    wealth = (1.0 + returns).cumprod()
    drawdown = wealth / wealth.cummax() - 1.0

    gross_performance_rows.append(
        {
            "portfolio": portfolio_name,
            "count": len(returns),
            "mean_daily_return": returns.mean(),
            "annualised_mean_return": (
                returns.mean() * 252
            ),
            "annualised_compound_return": (
                wealth.iloc[-1] ** (252 / len(returns))
                - 1.0
            ),
            "annualised_volatility": (
                returns.std() * np.sqrt(252)
            ),
            "sharpe_ratio": (
                sharpe_returns.mean()
                / sharpe_returns.std()
                * np.sqrt(252)
            ),
            "maximum_drawdown": drawdown.min(),
            "positive_fraction": (returns > 0).mean(),
        }
    )

gross_performance_summary = pd.DataFrame(
    gross_performance_rows
).set_index("portfolio")

gross_performance_summary

,count,mean_daily_return,annualised_mean_return,annualised_compound_return,annualised_volatility,sharpe_ratio,maximum_drawdown,positive_fraction
portfolio,,,,,,,,
Q1 low beta,2827,0.000467,0.117657,0.114432,0.136243,0.713704,-0.238021,0.548992
Q5 high beta,2827,0.001173,0.295562,0.285526,0.297318,0.925518,-0.429218,0.556774
Low minus high,2827,-0.000706,-0.177906,-0.194611,0.277011,-0.642233,-0.934491,0.466572
BAB capped,2827,-0.000238,-0.059958,-0.087521,0.251531,-0.238373,-0.727178,0.490626
BAB uncapped,2827,-0.000269,-0.067710,-0.098707,0.269177,-0.251546,-0.772395,0.490626


In [38]:
capm_dependent_returns = {
    "Q1 low beta": bab_construction_panel["q1_excess_return"],
    "Q5 high beta": bab_construction_panel["q5_excess_return"],
    "Low minus high": bab_construction_panel["low_minus_high_return"],
    "BAB capped": bab_construction_panel["bab_gross_return"],
    "BAB uncapped": bab_construction_panel["bab_uncapped_gross_return"],
}

capm_rows = []

for portfolio_name, portfolio_return in capm_dependent_returns.items():
    regression_data = pd.concat(
        [
            portfolio_return.rename("portfolio_return"),
            bab_construction_panel["spy_excess_return"],
        ],
        axis=1,
    ).dropna()

    explanatory_variables = sm.add_constant(regression_data["spy_excess_return"])

    model = sm.OLS(
        regression_data["portfolio_return"],
        explanatory_variables,
    ).fit(
        cov_type="HAC",
        cov_kwds={"maxlags": 5},
    )

    daily_alpha = model.params["const"]

    capm_rows.append(
        {
            "portfolio": portfolio_name,
            "count": int(model.nobs),
            "daily_alpha": daily_alpha,
            "annualised_alpha": daily_alpha * 252,
            "alpha_t_stat_hac": model.tvalues["const"],
            "realised_beta": (model.params["spy_excess_return"]),
            "beta_t_stat_hac": (model.tvalues["spy_excess_return"]),
            "r_squared": model.rsquared,
        }
    )

capm_regression_summary = pd.DataFrame(capm_rows).set_index("portfolio")

capm_regression_summary


,count,daily_alpha,annualised_alpha,alpha_t_stat_hac,realised_beta,beta_t_stat_hac,r_squared
portfolio,,,,,,,
Q1 low beta,2827,0.000137,0.034450,1.123824,0.499236,11.982796,0.421690
Q5 high beta,2827,0.000344,0.086730,2.230479,1.497936,54.534761,0.797469
Low minus high,2827,-0.000207,-0.052280,-0.836894,-0.998701,-15.375266,0.408305
BAB capped,2827,-0.000040,-0.010163,-0.141493,-0.395866,-7.780548,0.077808
BAB uncapped,2827,-0.000063,-0.015819,-0.206026,-0.412528,-7.619183,0.073780


## Conclusion

This experiment tested whether low-beta stocks offer superior raw or risk-adjusted returns within the S&P 100 universe.

### Methodology

- Estimated rolling 126-day stock betas relative to SPY.
- Ranked stocks cross-sectionally into beta quintiles.
- Examined 1-day and 5-day forward returns.
- Tested whether beta added information beyond realised volatility.
- Compared equal-weighted low-beta (Q1) and high-beta (Q5) portfolios.
- Constructed a beta-matched betting-against-beta (BAB) portfolio.
- Regularised portfolio betas using

  $$
  \widetilde{\beta}=0.6\widehat{\beta}+0.4,
  $$

  and capped gross exposure at $3\times$.
- Evaluated gross performance and CAPM alpha using HAC standard errors.

### Main findings

Forward returns increased rather than decreased with beta. Mean five-day returns rose from approximately **0.23%** in Q1 to **0.60%** in Q5, producing a high-minus-low spread of approximately **0.37%**.

The portfolio-level results confirmed this reversed relationship:

| Portfolio | Annualised mean return | Sharpe | Maximum drawdown | Realised beta | Annualised CAPM alpha | Alpha \(t\)-stat |
|---|---:|---:|---:|---:|---:|---:|
| Q1 low beta | 11.8% | 0.71 | -23.8% | 0.50 | 3.4% | 1.12 |
| Q5 high beta | 29.6% | 0.93 | -42.9% | 1.50 | 8.7% | 2.23 |
| Low minus high | -17.8% | -0.64 | -93.4% | -1.00 | -5.2% | -0.84 |
| BAB, capped | -6.0% | -0.24 | -72.7% | -0.40 | -1.0% | -0.14 |

Raw inverse-beta scaling was unstable because the estimated Q1 portfolio beta occasionally approached or crossed zero. Shrinkage and the \(3\times\) gross-exposure cap produced a numerically stable construction: median gross exposure was approximately **2.26×**, and the cap applied on approximately **12%** of dates.

However, the regularised BAB portfolio still retained a realised beta of approximately \(-0.40\). More importantly, CAPM adjustment did not reveal positive alpha. Its annualised alpha was approximately **-1.0%** and statistically insignificant.

### Interpretation and decision

The low-beta anomaly is not supported in this universe and sample. Low-beta stocks had neither higher raw returns nor a Sharpe advantage over high-beta stocks, while the BAB construction produced negative gross performance and no positive CAPM alpha.

Transaction-cost and additional robustness tests were not conducted because the strategy failed the gross-performance gating test. Costs could only weaken the result.

The positive performance of Q5 is recorded as a possible high-beta premium, but it is not promoted as a separate factor at this stage. Beta overlaps materially with realised volatility, and investigating the reversed signal would require a separate incremental, cost-aware study.

**Decision: reject low beta as a standalone alpha factor.**